## Anatomy of a Pod manifest

The smallest valid Pod is four required keys:

```yaml
apiVersion: v1
kind: Pod
metadata:
  name: hello
spec:
  containers:
    - name: app
      image: nginx:alpine
```

That runs. But real pods carry more. The realistic version, annotated:

```yaml
spec:
  containers:
    - name: app                     # required, unique within the pod
      image: nginx:1.27-alpine      # required
      imagePullPolicy: IfNotPresent # IfNotPresent | Always | Never
      ports:
        - containerPort: 80         # documentation only — opens nothing
          name: http                # lets Services reference it by name
      env:
        - name: GREETING
          value: hello
      command: ["nginx"]            # overrides the image ENTRYPOINT
      args: ["-g", "daemon off;"]   # overrides the image CMD
  restartPolicy: Always             # Always | OnFailure | Never
  terminationGracePeriodSeconds: 30 # SIGTERM-to-SIGKILL window
```

Four things worth calling out:

- **`containerPort` is documentation, not a firewall.** Listing it opens nothing — the container already listens where it likes. The field lets Services reference a port *by name* and tells humans what's exposed.
- **`command` overrides `ENTRYPOINT`, `args` overrides `CMD`** — same semantics as Docker, different words.
- **`imagePullPolicy` defaults are subtle.** Tag `:latest` or omitted → `Always`. Any other tag → `IfNotPresent`. This is why pinning a specific tag isn't just style — it changes behaviour.
- **A Deployment-managed pod gets an auto-suffix.** In notebook 03 you'll see names like `web-7c9d8f8b8d-x2k5p` — the random tail guarantees uniqueness across replicas.

On our map this is the **Pod** box in the Workloads band, zoomed to its **containers** — the spec fields you'll write most often in the whole course.